# L09 · LLM을 policy로 보기

## Goal

- token을 action으로 해석한다
- scalar와 verifiable reward를 구분한다
- reference KL의 역할을 설명한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L09:toy:42").hexdigest()
print(f"lesson=L09 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L09 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:07619f45c3d0f957828627af750a847b0eb5fcb65e4586c2552b7c6beec9efae data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: PPO + causal LM → **LLM policy·reward·reference KL** → RLHF/DPO/GRPO

$$r_t^{total}=r_t^{task}-\beta\left(\log\pi_\theta(a_t)-\log\pi_{ref}(a_t)\right)$$

prompt는 초기 state, 생성 token은 action, prefix는 다음 state입니다. scalar reward는 보통 response 끝에 붙지만 KL shaping은 action token마다 계산할 수 있습니다. reference model은 SFT 근처에서 policy가 너무 빨리 벗어나는 것을 측정합니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** action mask가 false인 세 번째 위치의 KL과 token reward는 얼마여야 하나요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>둘 다 0이어야 합니다. prompt/pad/tool token은 policy가 선택한 action이 아닙니다.</details>

In [2]:
from rl_study.algorithms.rlhf_ppo import compose_rlhf_rewards
policy_logp = torch.tensor([[-0.2, -0.3, 0.0]])
reference_logp = torch.tensor([[-0.3, -0.25, 0.0]])
token_action_mask = torch.tensor([[True, True, False]])
reward_parts = compose_rlhf_rewards(
    torch.tensor([1.0]), policy_logp, reference_logp,
    token_action_mask, kl_coefficient=0.1
)
print({"sampled_kl": reward_parts.sampled_kl.tolist(),
       "token_rewards": reward_parts.token_rewards.tolist(),
       "total_reward": reward_parts.total_rewards.tolist()})

{'sampled_kl': [[0.10000000894069672, -0.050000011920928955, 0.0]], 'token_rewards': [[-0.010000000707805157, 1.0049999952316284, -0.0]], 'total_reward': [0.9950000047683716]}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** sampled KL은 선택된 token 위의 log-ratio라 빠르지만 분산이 있고 음수가 될 수도 있습니다. full-distribution KL은 더 비싸지만 다른 진단 의미를 가집니다.

**흔한 함정:** scalar reward를 모든 token에 반복해서 더하면 response 길이만큼 보상을 복제합니다. terminal action에 한 번 붙이고 KL 항과 분리해 합계를 검산합니다. 회귀 test: `test_rlhf_reward_plus_token_kl_decomposition`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert reward_parts.token_rewards[0, 2].item() == 0.0
assert torch.allclose(reward_parts.total_rewards, reward_parts.token_rewards.sum(-1))
print("checks=passed")

checks=passed


**회상 문제:** sampled KL 한 항이 음수여도 전체 regularization이 잘못되었다고 단정할 수 없는 이유는 무엇인가요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** 세 번째 masked 위치의 값은 0이고 두 action token의 합계 reward는 0.995였습니다. task reward와 KL 비용을 따로 볼 수 있습니다.
- 실제 확인: `test_rlhf_reward_plus_token_kl_decomposition`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L10에서 SFT, reward source, rollout, PPO update를 하나의 lifecycle로 연결합니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

## Sources

- `learning-to-summarize-2020` — `docs/sources.yml`
- `instructgpt-2022` — `docs/sources.yml`